# Notebook 2 v2 — Safe setup and preflight

This workspace is deliberately separate from Notebook 1 and from your earlier Notebook 2 attempts.

**No package installation, uninstallation, Anaconda changes, checkpoint replacement, or Notebook 1 data modification is performed here.**

Run this notebook first. It only checks the existing project assets and creates folders under `Notebook_2_Improved_v2`.

In [1]:
from pathlib import Path
from datetime import datetime
import importlib.util, json, sys, platform, os

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
NOTEBOOK2_DIR = PROJECT_DIR / "Notebook_2_Improved_v2"

# Notebook 1 inputs: read-only
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"
GROUP_SEQUENCE_DIR = PROJECT_DIR / "data" / "processed" / "sequences_group_split_v1"
GROUP_SPLIT_PATH = GROUP_SEQUENCE_DIR / "video_split_manifest.csv"
ORIGINAL_KEYPOINT_DIR = PROJECT_DIR / "data" / "processed" / "keypoints_batch_all"
VITPOSE_SOURCE = PROJECT_DIR / "src" / "vitpose_model.py"
VITPOSE_CHECKPOINT = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_CANDIDATES = [PROJECT_DIR / "notebooks" / "yolo11n.pt", PROJECT_DIR / "yolo11n.pt"]
YOLO_WEIGHTS = next((p for p in YOLO_CANDIDATES if p.is_file()), None)

DIRS = {
    "notebooks": NOTEBOOK2_DIR / "notebooks",
    "src": NOTEBOOK2_DIR / "src",
    "manifests": NOTEBOOK2_DIR / "data" / "manifests",
    "tracking_pilot": NOTEBOOK2_DIR / "data" / "tracking_pilot_v2",
    "tracked_keypoints": NOTEBOOK2_DIR / "data" / "tracked_keypoints_v2",
    "tracked_sequences": NOTEBOOK2_DIR / "data" / "sequences_tracked_group_split_v2",
    "rgb_features": NOTEBOOK2_DIR / "data" / "rgb_features_r3d18_v2",
    "models": NOTEBOOK2_DIR / "outputs" / "models",
    "reports": NOTEBOOK2_DIR / "outputs" / "reports",
    "visualizations": NOTEBOOK2_DIR / "outputs" / "visualizations",
    "logs": NOTEBOOK2_DIR / "logs",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Notebook 2 v2:", NOTEBOOK2_DIR)
print("Python:", sys.executable)
print("Python version:", platform.python_version())
print("Manifest exists:", MANIFEST_PATH.is_file())
print("Original group-aware sequences exist:", GROUP_SEQUENCE_DIR.is_dir())
print("Original group split manifest exists:", GROUP_SPLIT_PATH.is_file())
print("Original NPZ directory exists:", ORIGINAL_KEYPOINT_DIR.is_dir())
print("Original NPZ count:", len(list(ORIGINAL_KEYPOINT_DIR.glob("*.npz"))) if ORIGINAL_KEYPOINT_DIR.is_dir() else 0)
print("ViTPose source exists:", VITPOSE_SOURCE.is_file())
print("ViTPose checkpoint exists:", VITPOSE_CHECKPOINT.is_file())
print("YOLO weights:", YOLO_WEIGHTS)

print("\nDependency availability (no installation is attempted):")
for name in ["numpy", "pandas", "cv2", "torch", "ultralytics", "sklearn", "torchvision", "PIL", "matplotlib"]:
    try:
        available = importlib.util.find_spec(name) is not None
    except Exception:
        available = False
    print(f"  {name}: {'available' if available else 'NOT FOUND'}")

assert PROJECT_DIR.is_dir(), f"Project folder not found: {PROJECT_DIR}"
assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"
assert GROUP_SPLIT_PATH.is_file(), f"Group split manifest not found: {GROUP_SPLIT_PATH}"
assert VITPOSE_SOURCE.is_file(), f"ViTPose source not found: {VITPOSE_SOURCE}"
assert VITPOSE_CHECKPOINT.is_file(), f"ViTPose checkpoint not found: {VITPOSE_CHECKPOINT}"
assert YOLO_WEIGHTS is not None, "YOLO weights yolo11n.pt not found in project notebooks or root."

import pandas as pd
split_df = pd.read_csv(GROUP_SPLIT_PATH)
required = {"video_path", "label", "split", "video_id", "recording_group_key"}
missing = required - set(split_df.columns)
assert not missing, f"Group split manifest is missing columns: {sorted(missing)}"

# Hard-check the frozen group split. This notebook does not create a new split.
groups = {s: set(split_df.loc[split_df["split"].eq(s), "recording_group_key"].astype(str)) for s in ["train", "val", "test"]}
for a,b in [("train","val"),("train","test"),("val","test")]:
    overlap = groups[a] & groups[b]
    assert not overlap, f"Recording-group overlap already exists in source split: {a} vs {b}: {list(overlap)[:5]}"

config = {
    "project_dir": str(PROJECT_DIR),
    "notebook2_dir": str(NOTEBOOK2_DIR),
    "manifest_path": str(MANIFEST_PATH),
    "group_split_path": str(GROUP_SPLIT_PATH),
    "original_keypoint_dir": str(ORIGINAL_KEYPOINT_DIR),
    "vitpose_source": str(VITPOSE_SOURCE),
    "vitpose_checkpoint": str(VITPOSE_CHECKPOINT),
    "yolo_weights": str(YOLO_WEIGHTS),
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "policy": "Notebook 1 is read-only. Notebook 2 v2 outputs are separate. No package changes."
}
(NOTEBOOK2_DIR / "notebook2_v2_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("\nGroup split rows:", len(split_df))
print("Classes:", split_df["label"].nunique())
print("Videos by split:\n", split_df.groupby("split").size().to_string())
print("PASS: Notebook 2 v2 workspace ready. No Notebook 1 files changed.")

Project: C:\Users\admin\ViTPose_Action_Recognition
Notebook 2 v2: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2
Python: C:\Users\admin\anaconda3\python.exe
Python version: 3.13.5
Manifest exists: True
Original group-aware sequences exist: True
Original group split manifest exists: True
Original NPZ directory exists: True
Original NPZ count: 13322
ViTPose source exists: True
ViTPose checkpoint exists: True
YOLO weights: C:\Users\admin\ViTPose_Action_Recognition\notebooks\yolo11n.pt

Dependency availability (no installation is attempted):
  numpy: available
  pandas: available
  cv2: available
  torch: available
  ultralytics: available
  sklearn: available
  torchvision: available
  PIL: available
  matplotlib: available

Group split rows: 13319
Classes: 101
Videos by split:
 split
test     2666
train    7999
val      2654
PASS: Notebook 2 v2 workspace ready. No Notebook 1 files changed.


## Run order

1. `01_pose_quality_audit.ipynb` — read-only inspection of the existing keypoint files.
2. `02_tracking_pilot.ipynb` — one-video YOLO + ByteTrack worker-process diagnostic.
3. `03_tracked_pose_extraction.ipynb` — resumable per-video YOLO + ByteTrack + ViTPose workers; pilot by default.
4. `04_build_tracked_sequences_group_split.ipynb` — builds new sequences using the exact existing recording-group assignments.
5. `05_train_stgcn.ipynb` — compact ST-GCN-inspired model; batch size 8; runs training in a worker process.
6. `06_extract_rgb_features.ipynb` — optional pretrained R3D-18 RGB features; pilot by default.
7. `07_train_rgb_pose_fusion.ipynb` — optional late RGB+pose fusion; requires complete features.
8. `08_compare_experiments.ipynb` — compare report files.

**Important limitation:** subprocess isolation prevents a worker failure from directly killing the Jupyter kernel, but it cannot guarantee that native libraries or Windows itself will never crash. The worker logs and exit codes will make failures diagnosable. No unsafe `KMP_DUPLICATE_LIB_OK` workaround is used.